# Preprocessing: raw I/Q traces to CSI features

Reads the raw Argos traces in `raw_traces/`, extracts the calibrated CSI (64 antennas × 52 subcarriers) and writes one feature file per cluster to `output_raw_full/clusters/`.

## Step 1: Settings

`FEATURE = "full"` keeps all 3328 amplitudes per frame and `NORM = "antenna"` divides each antenna by its own mean. `SAVE_CSI = True` also stores the complex CSI, which the robustness experiments and the Zheng et al. re-implementation need.

In [ ]:

import os
import json

import h5py
import numpy as np

RAW_DIR = "raw_traces"
OUT_DIR = "output_raw_full"

FEATURE = "full"
NORM = "antenna"
WINDOW = 50
STRIDE = 10
SKIP_FIRST = 200
CHUNK = 500
SAVE_CSI = True
CLUSTER_DIR = os.path.join(OUT_DIR, "clusters")

TRAIN_CLUSTERS = ["LOS_cluster1", "LOS_cluster2",
                  "NLOS_cluster1", "NLOS_cluster2", "NLOS_cluster3"]
TEST_CLUSTERS = ["LOS_cluster3", "LOS_cluster4",
                 "NLOS_cluster4", "NLOS_cluster5"]

def label_of(cluster):
    return 0 if cluster.startswith("NLOS") else 1

def file_of(cluster):
    return os.path.join(RAW_DIR, f"channe14_{cluster}.hdf5")

if NORM not in ("antenna", "frame", "none"):
    raise ValueError(f"unknown NORM: {NORM}")
if NORM == "antenna" and FEATURE in ("antenna", "both"):
    raise ValueError('NORM = "antenna" removes the antenna averages; use FEATURE = "full" or "subcarrier"')

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(CLUSTER_DIR, exist_ok=True)

## Step 2: Inspect a raw file

`Pilot_Samples` has shape (frames, 72 radio channels, 1120 samples, I/Q) and is stored as int16.

In [ ]:
with h5py.File(file_of("LOS_cluster1"), "r") as f:
    for k in f.keys():
        print(f"{k:18s} {f[k].shape}  {f[k].dtype}")
    print()
    for a in ["num_bs_ant", "num_bs_nodes", "num_mob_ant", "samples_per_user",
              "sample_dump", "pilot_length", "channel", "total_num_frames"]:
        print(f"{a:18s} {f.attrs[a]}")

## Step 3: Sample layout

Each 1120-sample record holds four 224-sample pilot slots and one noise-only slot. The 64-sample long training symbol starts at offset 47 of each slot. The measured user uses slot 0 in the LOS files and slot 3 in the NLOS files, and the calibration node uses the other one.

In [ ]:
LTS_FREQ = np.array([0,0,0,0,0,0,1,1,-1,-1,1,1,-1,1,-1,1,1,1,1,1,1,-1,-1,1,1,-1,1,-1,1,1,1,1,
                     0,1,-1,-1,1,1,-1,1,-1,1,-1,-1,-1,-1,-1,1,1,-1,-1,1,-1,1,-1,1,1,1,1,0,0,0,0,0], dtype=np.float32)
NULL_SUBCARRIERS = [0, 1, 2, 3, 4, 5, 32, 59, 60, 61, 62, 63]
LTS_OFFSET = 47
FFT_SIZE = 64
ANT_IDX = np.delete(np.arange(4, 72), [32, 33, 34, 35])

def slots_for(cluster):
    return (0, 3) if cluster.startswith("LOS") else (3, 0)

print("data subcarriers:", FFT_SIZE - len(NULL_SUBCARRIERS))
print("antennas kept:", len(ANT_IDX))

## Step 4: CSI extraction

FFT of the 64 LTS samples, multiplication by the known LTS, removal of the 12 null subcarriers and phase calibration with the calibration node, $H = H_{meas} / (H_{cal}/|H_{cal}|)$. The first `SKIP_FIRST` frames and the last frame of each cluster are dropped.

In [ ]:
def extract_csi(path, cluster, chunk=CHUNK, skip_first=SKIP_FIRST):
    meas_slot, cal_slot = slots_for(cluster)
    with h5py.File(path, "r") as f:
        ds = f["Pilot_Samples"]
        spu = int(f.attrs["samples_per_user"])
        n_frames = ds.shape[0]
        out = []
        for i in range(skip_first, n_frames - 1, chunk):
            block = ds[i:min(i + chunk, n_frames - 1)]
            per_slot = []
            for slot in (meas_slot, cal_slot):
                s = slot * spu + LTS_OFFSET
                iq = (block[:, :, s:s + FFT_SIZE, 0]
                      + 1j * block[:, :, s:s + FFT_SIZE, 1]).astype(np.complex64) * 2 ** -15
                h = np.fft.fftshift(np.fft.fft(iq, FFT_SIZE, axis=2), axes=2) * LTS_FREQ
                h = np.delete(h, NULL_SUBCARRIERS, axis=2)
                h = h[:, ::-1, :][:, ANT_IDX, :]
                per_slot.append(h)
            h_meas, h_cal = per_slot
            out.append(h_meas / (h_cal / np.abs(h_cal)))
    return np.concatenate(out)

## Step 5: Test on one cluster

In [ ]:
H_demo = extract_csi(file_of("NLOS_cluster2"), "NLOS_cluster2")
print("CSI shape:", H_demo.shape, H_demo.dtype)
print("mean |H|:", np.abs(H_demo).mean())

## Step 6: Amplitude and normalisation

The amplitude of every antenna is divided by its mean over the 52 subcarriers, which removes the absolute level and the location-dependent pattern across the antennas.

In [ ]:
def frame_features(H, feature=FEATURE, norm=NORM):
    amp = np.abs(H).astype(np.float32)
    if norm == "antenna":
        amp = amp / (amp.mean(axis=2, keepdims=True) + 1e-8)
    elif norm == "frame":
        amp = amp / (amp.mean(axis=(1, 2), keepdims=True) + 1e-8)

    if feature == "subcarrier":
        return amp.mean(axis=1)
    if feature == "antenna":
        return amp.mean(axis=2)
    if feature == "both":
        return np.concatenate([amp.mean(axis=1), amp.mean(axis=2)], axis=1)
    if feature == "full":
        return amp.reshape(len(amp), -1)
    raise ValueError(f"unknown feature: {feature}")

feats_demo = frame_features(H_demo)
print("features shape:", feats_demo.shape)

## Step 7: Quality check

Frames with non-finite values or only zeros are removed.

In [ ]:
def check_quality(name, x):
    bad = ~np.isfinite(x)
    zero_frames = np.all(x == 0, axis=1)
    if bad.any() or zero_frames.any():
        print(f"  WARNING {name}: {bad.sum()} non-finite values, "
              f"{zero_frames.sum()} all-zero frames (removed)")
    keep = ~(bad.any(axis=1) | zero_frames)
    return x[keep]

feats_demo = check_quality("NLOS_cluster2", feats_demo)
print("after check:", feats_demo.shape)
del H_demo, feats_demo

## Step 8: Process all nine clusters

In [ ]:
per_cluster, raw_level = {}, {}

for c in TRAIN_CLUSTERS + TEST_CLUSTERS:
    H = extract_csi(file_of(c), c)
    raw_level[c] = float(np.abs(H).mean())
    per_cluster[c] = check_quality(c, frame_features(H))
    if SAVE_CSI:
        np.save(os.path.join(CLUSTER_DIR, f"{c}_csi.npy"), H.astype(np.complex64))
    print(f"{c:14s} label={label_of(c)}  frames={len(per_cluster[c]):5d}  "
          f"features={per_cluster[c].shape[1]}  mean |H|={raw_level[c]:.3f}")
    del H

## Step 9: Save the per-cluster features

Features are saved before any z-score, because every split fits its own scaler on its training clusters. `clusters/index.json` lists the label, frame count and file names of every cluster.

In [ ]:
index = {}
for c, feats in per_cluster.items():
    fn = f"{c}_features.npy"
    np.save(os.path.join(CLUSTER_DIR, fn), feats.astype(np.float32))
    index[c] = {"label": label_of(c), "frames": int(len(feats)), "features_file": fn,
                "csi_file": f"{c}_csi.npy" if SAVE_CSI else None,
                "mean_amplitude_before_norm": raw_level[c]}

with open(os.path.join(CLUSTER_DIR, "index.json"), "w") as f:
    json.dump({"feature": FEATURE, "norm": NORM, "zscore_applied": False,
               "clusters": index}, f, indent=2)

for fn in sorted(os.listdir(CLUSTER_DIR)):
    print(f"{fn:32s} {os.path.getsize(os.path.join(CLUSTER_DIR, fn)) / 1e6:8.2f} MB")

## Step 10: Final checks

Checks the saved files and, if the dataset's `channel_data/` folder is available, compares the extracted CSI with the CSI processed by the dataset authors.

In [ ]:
with open(os.path.join(CLUSTER_DIR, "index.json")) as f:
    idx = json.load(f)["clusters"]
for c, info in idx.items():
    x = np.load(os.path.join(CLUSTER_DIR, info["features_file"]), mmap_mode="r")
    assert x.shape[0] == info["frames"]
    if info["csi_file"]:
        h = np.load(os.path.join(CLUSTER_DIR, info["csi_file"]), mmap_mode="r")
        assert h.shape == (info["frames"], 64, 52)
print("per-cluster files OK:", len(idx), "clusters")

In [ ]:
REF = os.path.join(os.path.dirname(RAW_DIR), "channel_data", "processed_channe14_NLOS_cluster2.hdf5")
if os.path.exists(REF):
    H = extract_csi(file_of("NLOS_cluster2"), "NLOS_cluster2")
    with h5py.File(REF, "r") as f:
        ref = f["mob_csi"][:].transpose(1, 0, 2)
    print("shapes:", H.shape, ref.shape)
    print("max |difference| vs channel_data:", np.abs(H - ref).max())
else:
    print("channel_data not found -- skipping comparison")